In [22]:
import os
import re
import json
import base64
import tiktoken
import time
import pandas as pd
import openai
from tqdm.notebook import tqdm
from openai import OpenAI
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()


True

In [23]:
# --- Verify Keys (Optional Check) ---
print("API Keys Set:")
print(f"Google API Key set: {'Yes' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"OpenAI API Key set: {'Yes' if os.environ.get('OPENAI_API_KEY') and os.environ['OPENAI_API_KEY'] != 'YOUR_OPENAI_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
# print(f"Anthropic API Key set: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') and os.environ['ANTHROPIC_API_KEY'] != 'YOUR_ANTHROPIC_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")


API Keys Set:
Google API Key set: No (REPLACE PLACEHOLDER!)
OpenAI API Key set: Yes


In [26]:
MODEL_LIMITS = {
    "gpt-3.5-turbo-0125": 16_385,
    "gpt-4-turbo-2024-04-09": 128_000,
    "gpt-4o-2024-05-13": 128_000,
    "gpt-4o-2024-11-20": 128_000,
    "gpt-5-mini-2025-08-07": 128_000,
    "gpt-5.2-2025-12-11": 128_000,
    "gpt-5-nano-2025-08-07": 128_000

}

# The cost per token for each model input.
MODEL_COST_PER_INPUT = {
    "gpt-3.5-turbo-0125": 0.0000005,
    "gpt-4-turbo-2024-04-09": 0.00001,
    "gpt-4o-2024-11-20": 0.0000025,
    "gpt-4o-2024-05-13": 0.000005,
    "gpt-5-mini-2025-08-07": 0.0000025,
    "gpt-5.2-2025-12-11": 0.00000175,
    "gpt-5-nano-2025-08-07": 0.00000005
}

# The cost per token for each model output.
MODEL_COST_PER_OUTPUT = {
    "gpt-3.5-turbo-0125": 0.0000015,
    "gpt-4-turbo-2024-04-09": 0.00003,
    "gpt-4o-2024-11-20":0.000010,
    "gpt-4o-2024-05-13": 0.000015,
    "gpt-5-mini-2025-08-07": 0.000002,
    "gpt-5.2-2025-12-11": 0.000014,
    "gpt-5-nano-2025-08-07": 0.0000004
}



In [ ]:
# If the question is a multi-choice question and you are unsure which one is correct, you must guess an option.  Please don't ask me any questions and give me the answer in the response.

def get_gpt_res(text, image, model):
    if image and model[:7] != "gpt-3.5":
        base64_image = encode_image(image)
        image_code = {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"}
        }
    
        response = client.chat.completions.create(
          model=model,
          messages=[
            {
              "role": "system",
              "content": [
                {
                  "type": "text",
                  "text": "You are a data analyst. I will give  you a background introduction and data analysis question. You must answer the question.  "
                }
              ]
            },
            {
              "role": "user",
              "content": [
                {
                  "type": "text",
                  "text": text
                },
                image_code
              ]
            }
          ],
          temperature=0,
          max_tokens=2256,
          top_p=1,
          frequency_penalty=0,
          presence_penalty=0
        )
    else:
        response = client.chat.completions.create(
          model=model,
          messages=[
            {
              "role": "system",
              "content": [
                {
                  "type": "text",
                  "text": "You are a data analyst. I will give  you a background introduction and data analysis question. You must answer the question. "
                }
              ]
            },
            {
              "role": "user",
              "content": [
                {
                  "type": "text",
                  "text": text
                }
              ]
            }
          ],
          temperature=0,
          # max_tokens=2256,
          top_p=1,
          frequency_penalty=0,
          presence_penalty=0
        )
    return response

In [28]:
samples = []
with open("./data_subset_olive.json", "r") as f:
    for line in f:
        samples.append(eval(line.strip()))
print(samples)

[{'id': '00000001', 'name': '2016-round-1-section-2-chip-off-the-old-block', 'url': 'https://www.eloquens.com/tool/vYDF7b/finance/modeloff-sample-past-questions/2016-round-1-section-2-chip-off-the-old-block', 'txt': '', 'questions': ['question6', 'question7', 'question8', 'question9', 'question10', 'question11', 'question12', 'question13', 'question14', 'question15', 'question16', 'question17', 'question18'], 'answers': ['D', 'D', 'I', 'A', 'H', 'C', 'E', 'H', 'D', 'A', 'I', 1661626, 323272], 'year': 2016}, {'id': '00000005', 'name': '2016-round-1-section-3-roll-the-dice', 'url': 'https://www.eloquens.com/tool/J1GcKr/finance/modeloff-sample-past-questions/2016-round-1-section-3-roll-the-dice', 'txt': '', 'questions': ['question20', 'question21', 'question22', 'question23', 'question24', 'question25', 'question26', 'question27', 'question28', 'question29'], 'answers': ['B', 'F', 'I', 'B', 'D', 'E', 'G', 'F', 178052, 23], 'year': 2016}, {'id': '00000006', 'name': '2016-round-2-section-3-

In [29]:
def gpt_tokenize(string: str, encoding) -> int:
    """Returns the number of tokens in a text string."""
    num_tokens = len(encoding.encode(string))
    return num_tokens

def find_jpg_files(directory):
    jpg_files = [file for file in os.listdir(directory) if file.lower().endswith('.jpg') or file.lower().endswith('.png')]
    return jpg_files if jpg_files else None

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')


def find_excel_files(directory):
    jpg_files = [file for file in os.listdir(directory) 
                 if (file.lower().endswith('xlsx') or file.lower().endswith('xlsb') or file.lower().endswith('xlsm')) 
                 and not "answer" in file.lower()
                 and not file.startswith('~$')]  # Exclude Excel temp files
    return jpg_files if jpg_files else None

def read_excel(file_path):
    # 读取Excel文件中的所有sheet
    try:
        if file_path.lower().endswith('.xlsb'):
            xls = pd.ExcelFile(file_path, engine='pyxlsb')
        else:
            # Use openpyxl for .xlsx and .xlsm files
            xls = pd.ExcelFile(file_path, engine='openpyxl')
        
        sheets = {}
        for sheet_name in xls.sheet_names:
            sheets[sheet_name] = xls.parse(sheet_name)
        return sheets
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        # Return empty dict so processing continues
        return {}

def dataframe_to_text(df):
    # 将DataFrame转换为文本
    text = df.to_string(index=False)
    return text

def combine_sheets_text(sheets):
    # 将所有sheet的文本内容组合起来
    combined_text = ""
    for sheet_name, df in sheets.items():
        sheet_text = dataframe_to_text(df)
        combined_text += f"Sheet name: {sheet_name}\n{sheet_text}\n\n"
    return combined_text

def read_txt(path):
    with open(path, "r") as f:
        return f.read()

def truncate_text(text, max_tokens=128000):
    # 计算当前文本的token数
    tokens = text.split()
    if len(tokens) > max_tokens:
        # 截断文本以确保不超过最大token数
        text = ' '.join(tokens[-max_tokens:])
    return text

### GPT 5 nano

In [ ]:
client = OpenAI()

tokens4generation = 6000
# model = "gpt-3.5-turbo-0125"
# model = "gpt-4-turbo-2024-04-09"
# model = "gpt-5-mini-2025-08-07"
# model = "gpt-5.2-2025-12-11"
model = "gpt-5-nano-2025-08-07"
model = "gpt-4o-2024-11-20"

data_path = "./data_subset_olive/"
total_cost = 0
encoding = tiktoken.encoding_for_model(model)
for id in tqdm(range(len(samples))):
    # print(sample)
    sample =samples[id]
    if len(sample["questions"]) > 0:
        start = sample["questions"][0]
        end = sample["questions"][-1]
        # print(start)
        # print(end)
        image = find_jpg_files(os.path.join(data_path, sample["id"]))
        if image:
            image = os.path.join(data_path, sample["id"], image[0])
        
        excel_content = ""
        excels = find_excel_files(os.path.join(data_path, sample["id"]))
        if excels:
            
            for excel in excels:
                excel_file_path = os.path.join(data_path,  sample["id"], excel)
                # print(excel_file_path)
                sheets = read_excel(excel_file_path)
                combined_text = combine_sheets_text(sheets)
                excel_content += f"The excel file {excel} is: " + combined_text

        introduction = read_txt(os.path.join(data_path, sample["id"], "introduction.txt"))
        questions = []
        for question_name in sample["questions"]:
            questions.append(read_txt(os.path.join(data_path, sample["id"], question_name+".txt")))
            
        # print(workbooks)
        
        text = ""
        if excel_content:
            text += f"The workbook is detailed as follows. {excel_content} \n"
        text += f"The introduction is detailed as follows. \n {introduction} \n"
        answers = []
        for question in questions:
            prompt = text +  f"The questions are detailed as follows. \n {question}"
        
            # print(len(encoding.encode(prompt)))
            cut_text = encoding.decode(encoding.encode(prompt)[tokens4generation-MODEL_LIMITS[model]:])
            # print(len(encoding.encode(prompt)))
            # print(prompt)
        # text = truncate_text(text, 20000)
            start = time.time()
            response = get_gpt_res(cut_text, image, model)
            cost = response.usage.completion_tokens * MODEL_COST_PER_OUTPUT[model] + response.usage.prompt_tokens * MODEL_COST_PER_INPUT[model]
            
            answers.append({"id": sample["id"], "model": response.model, "input": response.usage.prompt_tokens,
                            "output": response.usage.completion_tokens, "cost": cost, "time": time.time()-start, "response": response.choices[0].message.content})
            total_cost += cost
            print("Total cost: ", total_cost)
            # break
        save_path = os.path.join("./save_process", model)
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        with open(os.path.join(save_path, sample['id']+".json"), "w") as f:
            for answer in answers:
                json.dump(answer, f)
                f.write("\n")


  0%|          | 0/24 [00:00<?, ?it/s]

Total cost:  0.0012224999999999998
Total cost:  0.00310475
Total cost:  0.004744
Total cost:  0.00908065
Total cost:  0.018345399999999998
Total cost:  0.020054149999999996
Total cost:  0.021821649999999998
Total cost:  0.024065149999999997
Total cost:  0.0314123
Total cost:  0.0344685
Total cost:  0.04646155
Total cost:  0.0495687
Total cost:  0.0533424
Total cost:  0.0629038
Total cost:  0.06944639999999999
Total cost:  0.07612419999999999
Total cost:  0.0840908
Total cost:  0.0930954
Total cost:  0.1007232
Total cost:  0.109519
Total cost:  0.1194216
Total cost:  0.12775620000000001
Total cost:  0.13681400000000002
Total cost:  0.14119165000000003
Total cost:  0.14541420000000002
Total cost:  0.14955185000000001
Total cost:  0.15359920000000002
Total cost:  0.15727695000000003
Total cost:  0.16129705000000003
Total cost:  0.16661755000000003
Total cost:  0.17069375000000003
Total cost:  0.17578135000000003
Total cost:  0.18028180000000002
Total cost:  0.18372655000000002
Total cost:

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [ ]:
client = OpenAI()

tokens4generation = 6000
model = "gpt-5-nano-2025-08-07"
data_path = "./data_subset_olive/"
total_cost = 0
encoding = tiktoken.encoding_for_model(model)

# Create save path once at the beginning
save_path = os.path.join("./save_process", model)
if not os.path.exists(save_path):
    os.makedirs(save_path)

for id in tqdm(range(len(samples))):
    sample = samples[id]
    
    # Skip if already processed - CHECK THIS FIRST!
    if os.path.exists(os.path.join(save_path, sample['id']+".json")):
        print(f"Skipping {sample['id']} - already processed")
        continue
    
    if len(sample["questions"]) > 0:
        start = sample["questions"][0]
        end = sample["questions"][-1]
        
        image = find_jpg_files(os.path.join(data_path, sample["id"]))
        if image:
            image = os.path.join(data_path, sample["id"], image[0])
        
        excel_content = ""
        excels = find_excel_files(os.path.join(data_path, sample["id"]))
        if excels:
            for excel in excels:
                excel_file_path = os.path.join(data_path,  sample["id"], excel)
                sheets = read_excel(excel_file_path)
                combined_text = combine_sheets_text(sheets)
                excel_content += f"The excel file {excel} is: " + combined_text

        introduction = read_txt(os.path.join(data_path, sample["id"], "introduction.txt"))
        questions = []
        for question_name in sample["questions"]:
            questions.append(read_txt(os.path.join(data_path, sample["id"], question_name+".txt")))
        
        text = ""
        if excel_content:
            text += f"The workbook is detailed as follows. {excel_content} \n"
        text += f"The introduction is detailed as follows. \n {introduction} \n"
        answers = []
        for question in questions:
            prompt = text +  f"The questions are detailed as follows. \n {question}"
            cut_text = encoding.decode(encoding.encode(prompt)[tokens4generation-MODEL_LIMITS[model]:])
            
            start = time.time()
            response = get_gpt_res(cut_text, image, model)
            cost = response.usage.completion_tokens * MODEL_COST_PER_OUTPUT[model] + response.usage.prompt_tokens * MODEL_COST_PER_INPUT[model]
            
            answers.append({"id": sample["id"], "model": response.model, "input": response.usage.prompt_tokens,
                            "output": response.usage.completion_tokens, "cost": cost, "time": time.time()-start, "response": response.choices[0].message.content})
            total_cost += cost
            print("Total cost: ", total_cost)
        
        # Save immediately after processing each sample
        with open(os.path.join(save_path, sample['id']+".json"), "w") as f:
            for answer in answers:
                json.dump(answer, f)
                f.write("\n")

  0%|          | 0/24 [00:00<?, ?it/s]

Skipping 00000001 - already processed
Skipping 00000005 - already processed
Skipping 00000006 - already processed
Skipping 00000007 - already processed
Skipping 00000008 - already processed
Skipping 00000011 - already processed
Skipping 00000012 - already processed
Skipping 00000016 - already processed
Skipping 00000017 - already processed
Skipping 00000018 - already processed
Skipping 00000019 - already processed
Skipping 00000020 - already processed
Skipping 00000022 - already processed
Skipping 00000025 - already processed
Skipping 00000026 - already processed
Skipping 00000027 - already processed
Skipping 00000028 - already processed
Skipping 00000029 - already processed
Skipping 00000030 - already processed
Error reading ./data_subset_olive/00000032\~$MO14-Round-1-Dealing-With-Data-Workbook.xlsx: File is not a zip file
Total cost:  0.0067158
Total cost:  0.0131896
Total cost:  0.019708999999999997
Total cost:  0.0265964
Total cost:  0.033231
Total cost:  0.0407776
Total cost:  0.0

c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Total cost:  0.05592
Total cost:  0.060473799999999994
Total cost:  0.06476615
Total cost:  0.06996355
Total cost:  0.07601675
Total cost:  0.08165454999999999
Total cost:  0.086115
Total cost:  0.09058875
Total cost:  0.09772855
Total cost:  0.10475835
Total cost:  0.11236015
Total cost:  0.11935595
Total cost:  0.12641695
Total cost:  0.13462515
Total cost:  0.14168495
Total cost:  0.14934675
Total cost:  0.15663655
Total cost:  0.16504715
Total cost:  0.17297775
Total cost:  0.17986715
Total cost:  0.18740455
Total cost:  0.19536595
Total cost:  0.20248615
Total cost:  0.21056475


c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Total cost:  0.21182784999999998
Total cost:  0.21567935
Total cost:  0.2237499
Total cost:  0.22442215000000001
Total cost:  0.22698120000000002
Total cost:  0.22930385000000003
Total cost:  0.23584840000000004
Total cost:  0.23742310000000005
Total cost:  0.24296800000000005
Total cost:  0.25359890000000007
Total cost:  0.2633945000000001


In [21]:
cost_total = 0.2633945000000001 + 0.8508758999999999
print ("Final Total cost: $", cost_total)

Final Total cost: $ 1.1142704


37m 22.9s + 224m 47.4s

### GPT 4o

In [30]:
client = OpenAI()

tokens4generation = 6000
model = "gpt-4o-2024-11-20"
data_path = "./data_subset_olive/"
total_cost = 0
encoding = tiktoken.encoding_for_model(model)

# Create save path once at the beginning
save_path = os.path.join("./save_process", model)
if not os.path.exists(save_path):
    os.makedirs(save_path)

for id in tqdm(range(len(samples))):
    sample = samples[id]
    
    # Skip if already processed - CHECK THIS FIRST!
    if os.path.exists(os.path.join(save_path, sample['id']+".json")):
        print(f"Skipping {sample['id']} - already processed")
        continue
    
    if len(sample["questions"]) > 0:
        start = sample["questions"][0]
        end = sample["questions"][-1]
        
        image = find_jpg_files(os.path.join(data_path, sample["id"]))
        if image:
            image = os.path.join(data_path, sample["id"], image[0])
        
        excel_content = ""
        excels = find_excel_files(os.path.join(data_path, sample["id"]))
        if excels:
            for excel in excels:
                excel_file_path = os.path.join(data_path,  sample["id"], excel)
                sheets = read_excel(excel_file_path)
                combined_text = combine_sheets_text(sheets)
                excel_content += f"The excel file {excel} is: " + combined_text

        introduction = read_txt(os.path.join(data_path, sample["id"], "introduction.txt"))
        questions = []
        for question_name in sample["questions"]:
            questions.append(read_txt(os.path.join(data_path, sample["id"], question_name+".txt")))
        
        text = ""
        if excel_content:
            text += f"The workbook is detailed as follows. {excel_content} \n"
        text += f"The introduction is detailed as follows. \n {introduction} \n"
        answers = []
        for question in questions:
            prompt = text +  f"The questions are detailed as follows. \n {question}"
            cut_text = encoding.decode(encoding.encode(prompt)[tokens4generation-MODEL_LIMITS[model]:])
            
            start = time.time()
            response = get_gpt_res(cut_text, image, model)
            cost = response.usage.completion_tokens * MODEL_COST_PER_OUTPUT[model] + response.usage.prompt_tokens * MODEL_COST_PER_INPUT[model]
            
            answers.append({"id": sample["id"], "model": response.model, "input": response.usage.prompt_tokens,
                            "output": response.usage.completion_tokens, "cost": cost, "time": time.time()-start, "response": response.choices[0].message.content})
            total_cost += cost
            print("Total cost: ", total_cost)
        
        # Save immediately after processing each sample
        with open(os.path.join(save_path, sample['id']+".json"), "w") as f:
            for answer in answers:
                json.dump(answer, f)
                f.write("\n")

  0%|          | 0/24 [00:00<?, ?it/s]

Total cost:  0.049697500000000006
Total cost:  0.1023425
Total cost:  0.1529275
Total cost:  0.2052925
Total cost:  0.2578725
Total cost:  0.3133125
Total cost:  0.36926
Total cost:  0.4250675
Total cost:  0.4765375
Total cost:  0.53189
Total cost:  0.589135
Total cost:  0.647215
Total cost:  0.7051225
Total cost:  1.012065
Total cost:  1.3193375
Total cost:  1.6260100000000002
Total cost:  1.9334925000000003
Total cost:  2.2404050000000004
Total cost:  2.5477175000000005
Total cost:  2.8564500000000006
Total cost:  3.1658625000000007
Total cost:  3.4751950000000007
Total cost:  3.783727500000001
Total cost:  3.931832500000001
Total cost:  4.082412500000001
Total cost:  4.231197500000001
Total cost:  4.381257500000001
Total cost:  4.530907500000001
Total cost:  4.681305000000002
Total cost:  4.837362500000002
Total cost:  4.988025000000002
Total cost:  5.1395975000000025
Total cost:  5.291562500000002
Total cost:  5.443912500000002
Total cost:  5.5949500000000025
Total cost:  5.7469250

c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Total cost:  17.260602499999997
Total cost:  17.451534999999996
Total cost:  17.643284999999995
Total cost:  17.834487499999994
Total cost:  18.027989999999996
Total cost:  18.219082499999995
Total cost:  18.410207499999995
Total cost:  18.602177499999996
Total cost:  18.909889999999997
Total cost:  19.217062499999997
Total cost:  19.524295
Total cost:  19.8307675
Total cost:  20.1368
Total cost:  20.4475325
Total cost:  20.754015000000003
Total cost:  21.0606575
Total cost:  21.36815
Total cost:  21.6744225
Total cost:  21.981975
Total cost:  22.2897875
Total cost:  22.597849999999998


BadRequestError: Error code: 400

In [31]:
client = OpenAI()

tokens4generation = 6000
model = "gpt-4o-2024-11-20"
data_path = "./data_subset_olive/"
total_cost = 0
encoding = tiktoken.encoding_for_model(model)

# Create save path once at the beginning
save_path = os.path.join("./save_process", model)
if not os.path.exists(save_path):
    os.makedirs(save_path)

for id in tqdm(range(len(samples))):
    sample = samples[id]
    
    # Skip if already processed - CHECK THIS FIRST!
    if os.path.exists(os.path.join(save_path, sample['id']+".json")):
        print(f"Skipping {sample['id']} - already processed")
        continue
    
    if len(sample["questions"]) > 0:
        start = sample["questions"][0]
        end = sample["questions"][-1]
        
        image = find_jpg_files(os.path.join(data_path, sample["id"]))
        if image:
            image = os.path.join(data_path, sample["id"], image[0])
        
        excel_content = ""
        excels = find_excel_files(os.path.join(data_path, sample["id"]))
        if excels:
            for excel in excels:
                excel_file_path = os.path.join(data_path,  sample["id"], excel)
                sheets = read_excel(excel_file_path)
                combined_text = combine_sheets_text(sheets)
                excel_content += f"The excel file {excel} is: " + combined_text

        introduction = read_txt(os.path.join(data_path, sample["id"], "introduction.txt"))
        questions = []
        for question_name in sample["questions"]:
            questions.append(read_txt(os.path.join(data_path, sample["id"], question_name+".txt")))
        
        text = ""
        if excel_content:
            text += f"The workbook is detailed as follows. {excel_content} \n"
        text += f"The introduction is detailed as follows. \n {introduction} \n"
        answers = []
        for question in questions:
            prompt = text +  f"The questions are detailed as follows. \n {question}"
            cut_text = encoding.decode(encoding.encode(prompt)[tokens4generation-MODEL_LIMITS[model]:])
            
            start = time.time()
            response = get_gpt_res(cut_text, image, model)
            cost = response.usage.completion_tokens * MODEL_COST_PER_OUTPUT[model] + response.usage.prompt_tokens * MODEL_COST_PER_INPUT[model]
            
            answers.append({"id": sample["id"], "model": response.model, "input": response.usage.prompt_tokens,
                            "output": response.usage.completion_tokens, "cost": cost, "time": time.time()-start, "response": response.choices[0].message.content})
            total_cost += cost
            print("Total cost: ", total_cost)
        
        # Save immediately after processing each sample
        with open(os.path.join(save_path, sample['id']+".json"), "w") as f:
            for answer in answers:
                json.dump(answer, f)
                f.write("\n")

  0%|          | 0/24 [00:00<?, ?it/s]

Skipping 00000001 - already processed
Skipping 00000005 - already processed
Skipping 00000006 - already processed
Skipping 00000007 - already processed
Skipping 00000008 - already processed
Skipping 00000011 - already processed
Skipping 00000012 - already processed
Skipping 00000016 - already processed
Skipping 00000017 - already processed
Skipping 00000018 - already processed
Skipping 00000019 - already processed
Skipping 00000020 - already processed
Skipping 00000022 - already processed
Skipping 00000025 - already processed
Skipping 00000026 - already processed
Skipping 00000027 - already processed
Skipping 00000028 - already processed
Skipping 00000029 - already processed
Skipping 00000030 - already processed
Skipping 00000032 - already processed
Skipping 00000033 - already processed
Total cost:  0.30656250000000007
Total cost:  0.6133450000000001
Total cost:  0.9206975000000002
Total cost:  1.2273900000000002
Total cost:  1.5330025000000003
Total cost:  1.8404250000000002
Total cos

c:\Users\ustra\miniconda3\envs\dsbench\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Total cost:  4.9312075
Total cost:  4.946845000000001
Total cost:  4.959065000000001
Total cost:  4.970320000000001
Total cost:  4.990375000000001
Total cost:  4.9972400000000015
Total cost:  5.0093200000000015
Total cost:  5.017317500000002
Total cost:  5.028675000000002
Total cost:  5.043722500000001
Total cost:  5.055115000000002


In [32]:
cost_total = 5.055115000000002 + 22.597849999999998
print ("Final Total cost: $", cost_total)

Final Total cost: $ 27.652965


7m 10.5s + 49m 58.1s